In [4]:
###########################################################################
## Basic stuff
###########################################################################
%load_ext autoreload
%autoreload
from IPython.core.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))
display(HTML("""<style>div.output_area{max-height:10000px;overflow:scroll;}</style>"""))


###########################################################################
## Music
###########################################################################
from myMusicPathData import myMusicPathData


###########################################################################
## Utils
###########################################################################
from timeUtils import timestat
from fileIO import fileIO
from fsUtils import fsPath, fsInfo, dirUtil, fileUtil
from pandas import DataFrame, Series


###########################################################################
## DB
###########################################################################
from masterManualEntries import masterManualEntries
from masterManualEntriesUtils import masterManualEntriesUtils
from masterArtistNameDB import masterArtistNameDB
from mainDB import mainDB
from matchArtistToDB import masterMatch

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# General

In [5]:
manDB      = masterArtistNameDB("main")
multimanDB = masterArtistNameDB("multi")
io   = fileIO()
mmpd = myMusicPathData()
mme  = masterManualEntries()
meu  = masterManualEntriesUtils()
#maindb.setMasterDBData()   ### Full DB Access

========================= masterArtistNameDB("main") =========================
Current Time is Thu Dec 16, 2021 08:58 for Getting Manual Renames Data From Main Pickle File
Process [Getting Manual Renames Data From Main Pickle File] Took 0.1 Seconds
  No duplicate key/values in manual renames
  No recursive key/values in manual renames
masterArtistNameDB("main") Summary:
  Entries: 45021
  Artists: 34171
========================= masterArtistNameDB("multi") =========================
Current Time is Thu Dec 16, 2021 08:58 for Getting Manual Renames Data From Main Pickle File
Process [Getting Manual Renames Data From Main Pickle File] Took 0.0 Seconds
  No duplicate key/values in manual renames
  No recursive key/values in manual renames
masterArtistNameDB("multi") Summary:
  Entries: 705
  Artists: 644
========================= myMusicPathData(/Volumes/Piggy/Music/Matched) =========================
========================= masterManualEntries(install=False) =========================
===

# Find My Music

In [ ]:
mmpdData = mmpd.findMyMusic()

In [ ]:
mmpd.saveData(artistAlbums=mmpdData, local=False)

In [ ]:
#mmpdData = mmpd.getData()
mmpd.getSummary(mmpdData).head()

# Match To Global Manual Music DB

In [ ]:
mme      = masterManualEntries()
mmeDF    = mme.getData()
mmpdData = Series(mmpd.getData())
mm       = masterMatch()

In [ ]:
def nearMatches(x, cutoff=0.9):
    artistNameUpper = x.upper()
    searchResults = masterArtistNames.apply(mm.getLevenshtein, x2=artistNameUpper)
    return searchResults[searchResults > cutoff]

mmpdData           = Series(mmpd.getData())
masterArtistNames  = mmeDF["ArtistName"]
myMusicArtistNames = Series(mmpdData.index)

In [ ]:
ts = timestat("Finding Near Matches For {0} Unmatched Master Artists Using {1} Unknown List Artists".format(len(myMusicArtistNames), len(masterArtistNames)))
N = len(myMusicArtistNames)
matchResults = {}
for i,(idx,artistName) in enumerate(myMusicArtistNames.iteritems()):
    matchResults[artistName] = nearMatches(artistName)
    if (i+1) % 250 == 0 or (i+1) == 50:
        ts.update(n=i+1, N=N)
        break
#myMusicArtistNameMatches = myMusicArtistNames.apply(nearMatches)
ts.stop()
#print("Found {0} Possibly Matched Artists".format(aDFNear.sum()))

In [ ]:
results = masterArtistNames.apply(mm.getLevenshtein, x2="A Flock Of Seagulls")

In [ ]:
### Perfect Match
perfectMatches = {artistName: mmeDF[mmeDF["ArtistName"] == artistName] for idx,artistName in myMusicArtistNames.iteritems()}

In [ ]:
myMusicMatchIDs  = {}
noMusicMatchs    = {}
multiMusicMatchs = {}
for artistName,artistMatches in perfectMatches.items():
    if artistMatches.shape[0] == 1:
        myMusicMatchIDs[artistName] = list(artistMatches.index)[0]
    elif artistMatches.shape[0] > 1:
        print("Multiple Names: {0}".format(artistName))
        multiMusicMatchs[artistName] = artistMatches
    else:
        print("No Matches: {0}".format(artistName))
        noMusicMatchs[artistName] = True
        
myMusicMatchIDs  = Series(myMusicMatchIDs)
noMusicMatchs    = Series(noMusicMatchs)
multiMusicMatchs = Series(multiMusicMatchs)

In [ ]:
from masterDBData import masterDBData
mdbData = masterDBData(dtype="Search")
mdbData.loadArtists()
mdbData.loadAlbums()

In [ ]:
from listUtils import getFlatList

def getMasterAlbums(row):
    def getArtistDBAlbums(db,dbID):
        albums = mdbData.getArtistDBAlbumsFromID(db,dbID)
        retval = albums if albums is not None else []
        return retval    
    
    fullAlbumsList = getFlatList([getArtistDBAlbums(db,dbID) for db,dbID in row[row.notna()].iteritems()])
    retval = list(Series(fullAlbumsList).unique()) if len(fullAlbumsList) > 0 else []
    return retval

ts = timestat("Getting Master Artist Albums For {0} Entries".format(mmeDF.shape[0]))
masterArtistAlbums = mmeDF[mdbData.dbs].apply(getMasterAlbums, axis=1)
ts.stop()

In [ ]:
mmeDF["Albums"] = masterArtistAlbums

In [ ]:
masterArtistAlbumsData = mmeDF[["ArtistName", "Albums"]]

In [ ]:
masterArtistAlbumsData["NumAlbums"] = masterArtistAlbumsData["Albums"].apply(len)

In [ ]:
io.save(idata=masterArtistAlbums, ifile="masterArtistAlbums.p")

## Match My Music

In [7]:
%autoreload
from matchRunParams import matchRunParams
from masterMatchGate import masterMatchGate
from matchDBData import matchDBData
from masterDBData import masterDBData

### Create My Music DB

In [8]:
dbToUse = "MyMusic"

io = fileIO()
artistSummaryDF     = io.get("../music/music_artistSummaryDF.p")
myMusicArtistAlbums = DataFrame({artistName: {"ArtistName": artistName, "Albums": df["AlbumName"].to_list()} for artistName,df in artistSummaryDF.groupby("ArtistName")}).T
myMusicArtistAlbums["NumAlbums"] = myMusicArtistAlbums["Albums"].apply(len)

mdbDataDBToUse    = masterDBData(dtype="User", dbs="MyMusic")
mdbDataDBToUse.setUserDBData("MyMusic", myMusicArtistAlbums["ArtistName"], myMusicArtistAlbums["NumAlbums"], myMusicArtistAlbums["Albums"])
artistDBToUseData = mdbDataDBToUse.getDBBasicInfo(dbToUse)

========================= MasterDB Data(dtype=User, dbs=MyMusic) =========================


### Create Master Music DB

In [3]:
from masterManualEntriesUtils import masterManualEntriesUtils
meu = masterManualEntriesUtils()
masterArtistAlbumsData = meu.getDBDF()

========================= masterManualEntries(install=False) =========================
Current Time is Wed Dec 15, 2021 20:35 for Getting Manual Entries Data From Main Pickle File
Process [Getting Manual Entries Data From Main Pickle File] Took 0.7 Seconds
========================= MasterDB Data(dtype=Search, dbs=None) =========================
Current Time is Wed Dec 15, 2021 20:35 for Getting Manual Entries Data From Main Pickle File
Process [Getting Manual Entries Data From Main Pickle File] Took 8.9 Seconds


In [5]:
mdbData = masterDBData(dtype="User", dbs="Master")
mdbData.setUserDBData("Master", masterArtistAlbumsData["ArtistName"], masterArtistAlbumsData["NumAlbums"], masterArtistAlbumsData["Albums"])

matchData = matchDBData(mmeDF=None, dtype="User", dbs="Master")
matchData.setMDBData(mdbData)

========================= MasterDB Data(dtype=User, dbs=Master) =========================
========================= matchDBData(dtype=User, dbs=Master) =========================
  Not Hashing Out Previously Matched dbIDs
  Set mdbData with DBs: ['Master']


In [9]:
%autoreload
from matchRunParams import matchRunParams
from masterMatchGate import masterMatchGate
from matchDBData import matchDBData
from masterDBData import masterDBData
from matchArtistToDB import matchArtistToDB

print("Full: {0}".format(artistDBToUseData.shape[0]))
idxs = artistDBToUseData["NumAlbums"] > 0
artistsToMatchData = artistDBToUseData.loc[idxs]
print(" Use: {0}".format(artistsToMatchData.shape[0]))

mrp = matchRunParams()
mmg = masterMatchGate(mdbDataDBToUse, dbToUse)
mmg.setArtistToMatchData(artistsToMatchData)

Full: 5170
 Use: 5170
========================= masterMatchGate(mdbDataDBToUse, dbToUse=MyMusic) =========================
  Setting Artist/Album Match Data
	Artists             : 5170
	Max Albums          : 1478
	Min Albums          : 1


In [10]:
tsMatch = timestat("Matching [{0}] Artists From [{1}]".format(artistsToMatchData.shape[0], dbToUse))

Current Time is Wed Dec 15, 2021 20:37 for Matching [5170] Artists From [MyMusic]


In [11]:
from matchUtils import getUnmatchedArtistsFromDB, getArtistsToMatch, serialRun, multiProcRun
usePool = True
numProcs=3

numRunData = mmg.setArtistRunData()
tsName = timestat("Matching Artist Names")
print("  Matching {0} Artists".format(numRunData))
params={"artistCutoff": 0.85, "by": "Artist", 'matchData': matchData}
runResults = multiProcRun(mmg, numProcs=numProcs, params=params) if usePool else serialRun(mmg=mmg, params=params)
mmg.setMatchedArtistData(runResults)
mmg.updateArtistDataArtistRunStatus()
tsName.stop()
 
tsMatch.update()

Current Time is Wed Dec 15, 2021 20:37 for Matching Artist Names
  Matching {'ForRun': 5170, 'ForSubRun': 5170} Artists
Process [Matching Artist Names] Took 9.8 Minutes
Process [Matching [5170] Artists From [MyMusic]] Has Run For 9.8 Minutes


In [13]:
## Load Albums For Matched Names

tsIDX = timestat("Loading Album Data For Matched Index")
#matchData.loadAlbums(idxReq=mmg.matchedArtistIDXReq)
tsIDX.stop()
tsMatch.update()

Current Time is Wed Dec 15, 2021 20:50 for Loading Album Data For Matched Index
Process [Loading Album Data For Matched Index] Took 0.0 Seconds
Process [Matching [5170] Artists From [MyMusic]] Has Run For 13.4 Minutes


In [14]:
## Match Artist Albums

%autoreload
from matchArtistToDB import matchArtistToDB
tsAlbum = timestat("Matching Artist Albums")
for runNum,(runDefs, runThresholds) in enumerate(mrp.getRunParams()):
    #print("="*125)
    #print("Run #: {0}".format(runNum))
    runThresholds['albumCutoff'] = 0.7
    runThresholds['numAlbums']   = 1
    runThresholds['score']       = 0.0
    runThresholds["by"]          = "Album"
    runThresholds['detail']      = 0
    runThresholds['matchData']   = matchData
    runStatus = True
    while runStatus is True:
        numArtistsToMatchForRun = mmg.setAlbumRunData(**runDefs)
        if numArtistsToMatchForRun["ForRun"] <= 0:
            runStatus = False
            continue
            
        print("  Matching {0} Artists For Run Number [{1}]".format(numArtistsToMatchForRun, runNum))        
        runResults = multiProcRun(mmg=mmg, numProcs=numProcs, params=runThresholds) if usePool else serialRun(mmg=mmg, params=runThresholds)
        mmg.setMatchedArtistAlbumData(runResults)
        mmg.updateArtistDataAlbumRunStatus()
tsAlbum.stop()
print("")

Current Time is Wed Dec 15, 2021 20:50 for Matching Artist Albums
  Matching {'ForRun': 1, 'ForSubRun': 1} Artists For Run Number [0]
  Matching {'ForRun': 17, 'ForSubRun': 17} Artists For Run Number [1]
  Matching {'ForRun': 396, 'ForSubRun': 396} Artists For Run Number [2]
  Matching {'ForRun': 982, 'ForSubRun': 982} Artists For Run Number [3]
  Matching {'ForRun': 1553, 'ForSubRun': 1553} Artists For Run Number [4]
  Matching {'ForRun': 2221, 'ForSubRun': 2221} Artists For Run Number [5]
Process [Matching Artist Albums] Took 4.0 Minutes



In [15]:
def getArtistMatchResults(matchedArtistAlbumData):
    dbToUseResults = {}
    for dbToUseID,dbToUseArtistMatchData in matchedArtistData.items():
        dbToUseResults[dbToUseID] = {}
        for db,dbArtistMatchData in dbToUseArtistMatchData.items():
            if dbArtistMatchData is None:
                continue
            artistResult = DataFrame(dbArtistMatchData).T
            dbIDMatch    = artistResult[(artistResult["Levenshtein"] >= 0.8)].sort_values(by=["Levenshtein"], ascending=False).head(1)
            if dbIDMatch.shape[0] > 0:
                dbToUseResults[dbToUseID][db] = dbIDMatch.index[0]
    dbToUseResults = Series(dbToUseResults).apply(Series)
    return dbToUseResults

def getAlbumMatchResults(matchedArtistAlbumData):
    dbToUseResults = {}
    for dbToUseID,dbToUseAlbumMatchData in matchedArtistAlbumData.items():
        dbToUseResults[dbToUseID] = {}
        for db,dbAlbumMatchData in dbToUseAlbumMatchData.items():
            if dbAlbumMatchData is None:
                continue
            albumResult = DataFrame(dbAlbumMatchData).T
            dbIDMatch   = albumResult[(albumResult["Match"] >= 1.0)].sort_values(by=["Match","Score"], ascending=False).head(1)
            if dbIDMatch.shape[0] > 0:
                dbToUseResults[dbToUseID][db] = dbIDMatch.index[0]
    dbToUseResults = Series(dbToUseResults).apply(Series)
    return dbToUseResults

In [16]:
matchedArtistData      = {item["ArtistID"]: item["Results"] for item in mmg.matchedArtistData}
matchedArtistAlbumData = {item["ArtistID"]: item["Results"] for item in mmg.matchedArtistAlbumData}
dbToUseResultsArtists  = getArtistMatchResults(matchedArtistData)
dbToUseResultsAlbums   = getAlbumMatchResults(matchedArtistAlbumData)

In [21]:
io.save(idata=dbToUseResultsArtists[dbToUseResultsArtists.Master.isna()], ifile="myMusicArtistsNotMatched.p")
io.save(idata=dbToUseResultsAlbums[dbToUseResultsAlbums.Master.isna()], ifile="myMusicArtistAlbumssNotMatched.p")

In [25]:
for artist in dbToUseResultsArtists[dbToUseResultsArtists["Master"].isna()]["Master"].head(300).tail(300).index:
    print("meu.newArtist(\"{0}\",  Discogs=\"\",  RateYourMusic=\"\",  MusicBrainz=\"\")".format(artist))

meu.newArtist("Alan Hartwell Big Band",  Discogs="",  RateYourMusic="",  MusicBrainz="")
meu.newArtist("All Chrome",  Discogs="",  RateYourMusic="",  MusicBrainz="")
meu.newArtist("Another Nothing",  Discogs="",  RateYourMusic="",  MusicBrainz="")
meu.newArtist("Anton Ishutin",  Discogs="",  RateYourMusic="",  MusicBrainz="")
meu.newArtist("Apothys",  Discogs="",  RateYourMusic="",  MusicBrainz="")
meu.newArtist("Bad Times",  Discogs="",  RateYourMusic="",  MusicBrainz="")
meu.newArtist("Bebe Rebozo",  Discogs="",  RateYourMusic="",  MusicBrainz="")
meu.newArtist("Belva Plane",  Discogs="",  RateYourMusic="",  MusicBrainz="")
meu.newArtist("Bent Leg Fatima",  Discogs="",  RateYourMusic="",  MusicBrainz="")
meu.newArtist("Blastula",  Discogs="",  RateYourMusic="",  MusicBrainz="")
meu.newArtist("Blek Ink",  Discogs="",  RateYourMusic="",  MusicBrainz="")
meu.newArtist("Bugs In Amber",  Discogs="",  RateYourMusic="",  MusicBrainz="")
meu.newArtist("Busytoby",  Discogs="",  RateYourMusic=

In [6]:
meu.newArtist("Alan Hartwell Big Band",  Discogs="4846670",  AllMusic="0001058980")
meu.newArtist("All Chrome",  Discogs="556969",  MusicBrainz=meu.getmbid("https://musicbrainz.org/artist/2c290c49-d83d-4a08-a229-05d8a63eaf6b"))
meu.newArtist("Another Nothing",  Discogs="729247",  MusicBrainz=meu.getmbid("https://musicbrainz.org/artist/ae48af03-c539-4c77-9e60-f18020e36b0d"))
meu.newArtist("Anton Ishutin",  Discogs="2054079",  Deezer="445230",  MusicBrainz=meu.getmbid("https://musicbrainz.org/artist/8ae87648-898a-4a75-97a9-3c4ef7bfe0aa"))
meu.newArtist("Bad Times",  Discogs="1461388",  MusicBrainz=meu.getmbid("https://musicbrainz.org/artist/05f97d2e-650e-430f-b238-fa186a7c74c4"))
meu.newArtist("Bebe Rebozo",  Discogs="969446",  MusicBrainz=meu.getmbid("https://musicbrainz.org/artist/616df4f3-e595-435a-ba9f-29aba82e880b"))
meu.newArtist("Belva Plane",  Discogs="4882775",  MusicBrainz=meu.getmbid("https://musicbrainz.org/artist/ac49f2b6-f0ca-45e9-b269-a89cdf4004cc"))
meu.newArtist("Bent Leg Fatima",  Discogs="378862",  MusicBrainz=meu.getmbid("https://musicbrainz.org/artist/f56bca08-4ea3-4210-b7cd-abb898dfd54a"))
meu.newArtist("Blastula",  Discogs="1015890",  MusicBrainz=meu.getmbid("https://musicbrainz.org/artist/fa3552e1-f9e1-41fc-a884-14c6797ba939"))
meu.newArtist("Blek Ink",  Discogs="637924",  AllMusic="0000759764")
meu.newArtist("Bugs In Amber",  Discogs="3419661",  MusicBrainz=meu.getmbid("https://musicbrainz.org/artist/d030494b-367e-4df5-8e46-6bf07989029a"), AllMusic="0000939629")
meu.newArtist("Busytoby",  Discogs="873167",  MusicBrainz=meu.getmbid("https://musicbrainz.org/artist/dd861186-a0ae-4f29-b59f-d38bab83bd08"))
meu.newArtist("Castigate",  Discogs="1004366",  MusicBrainz=meu.getmbid("https://musicbrainz.org/artist/643bdaf1-65c4-43c2-b6c8-a334752ee20c"))
meu.newArtist("Chicken Coupe DeVille",  Discogs="4311648",  MusicBrainz=meu.getmbid("https://musicbrainz.org/artist/837cb2fd-202a-4cbe-a429-c332629546aa"))
meu.newArtist("Children's Television Workshop",  Discogs="379491",  MusicBrainz=meu.getmbid("https://musicbrainz.org/artist/b9563601-6557-44fb-b041-21adcb59f508"))
meu.newArtist("Christine Perfect",  Discogs="304393",  MusicBrainz=meu.getmbid("https://musicbrainz.org/artist/21551f7d-93f2-4cb7-ba71-7f260058e923"), AllMusic="0000124031")
meu.newArtist("Clark Gesner",  Discogs="1859108",  MusicBrainz=meu.getmbid("https://musicbrainz.org/artist/3e6c7d25-67de-4569-8401-b1a64d3591be"))

  ==> Added New Row [ArtistName    Alan Hartwell Big Band
Discogs                      4846670
AllMusic                  0001058980
Name: 32d7027d-4628-45e7-967c-ef58d064853a, dtype: object]
  ==> Added New Row [ArtistName                                  All Chrome
Discogs                                         556969
MusicBrainz    161216463295732778342684647755035425086
Name: 87c7786b-7672-4b16-a835-6591b4cec71f, dtype: object]
  ==> Added New Row [ArtistName                             Another Nothing
Discogs                                         729247
MusicBrainz    151777408525610896347338518283423662952
Name: 16ee48d6-ddee-44be-b536-5a1097685298, dtype: object]
  ==> Added New Row [ArtistName                               Anton Ishutin
Discogs                                        2054079
Deezer                                          445230
MusicBrainz    177145890774834782935969436789409458672
Name: ed6d3262-075e-40b9-9e9e-8d48dc7bf18d, dtype: object]
  ==> Added New Row

In [7]:
meu.saveDF()

Current Time is Thu Dec 16, 2021 09:16 for Saving Manual Entries Data To Main Pickle File
Process [Saving Manual Entries Data To Main Pickle File] Took 1.9 Seconds


In [8]:
meu      = masterManualEntriesUtils()
meu.newArtist("Cowboy & Spin Girl",  Discogs="632402",  MusicBrainz=meu.getmbid("https://musicbrainz.org/artist/6b8225f3-bbd2-4a68-b42b-5742a4309d0f"), AllMusic="0000101460")
meu.newArtist("Coyabalites",  Discogs="1511252",  MusicBrainz=meu.getmbid("https://musicbrainz.org/artist/e69a70fb-f631-463e-89b3-533825c4c767"))
meu.newArtist("DFL",  Discogs="251993",  MusicBrainz=meu.getmbid("https://musicbrainz.org/artist/adbd9e9c-8e6e-414a-a6cf-1bc3436ceacb"), RateYourMusic="21820")
meu.newArtist("DJ Spinatik",  Discogs="1705874",  MusicBrainz=meu.getmbid("https://musicbrainz.org/artist/bdd02028-4c55-4b56-9c38-7e5c13e5e679"))
meu.newArtist("Dave Myers & The Surftones",  Discogs="254863",  AllMusic="0000590067")
meu.newArtist("Davie Allan & The Arrows",  Discogs="246649",  AllMusic="0000191458")
meu.newArtist("Divine Regale",  Discogs="1470637",  AllMusic="0000172913")
meu.newArtist("Emiliana Cantone",  Discogs="4674399",  AllMusic="0002456085")
meu.newArtist("Eric Michael Gillett",  Discogs="2086616",  AllMusic="0000192780")
meu.newArtist("Ex-Fork",  Discogs="1402721",  MusicBrainz=meu.getmbid("https://musicbrainz.org/artist/e47466e3-102c-4412-9b62-a6dd222c8252"))
meu.newArtist("Fluke Starbucker",  Discogs="881040",  MusicBrainz=meu.getmbid("https://musicbrainz.org/artist/bd378af8-bfaa-4ebf-84a1-cac686cddfac"))
meu.newArtist("Fortydaysrain",  Discogs="1431853",  MusicBrainz=meu.getmbid("https://musicbrainz.org/artist/90c97a3b-6bab-4a5b-96a9-7a35c90bc40b"))
meu.newArtist("Friends Of Sound",  Discogs="996911",  AllMusic="0001373424")
meu.newArtist("From Safety To Where",  Discogs="960466",  AllMusic="0000988118")
meu.newArtist("Funny Looking Kids",  Discogs="1289382",  AllMusic="0001936650")
meu.newArtist("Fury For Another",  Discogs="1607944",  AllMusic="0000801451")
meu.newArtist("Gabe Nieto",  Discogs="7741610",  MusicBrainz=meu.getmbid("https://musicbrainz.org/artist/e4b6df11-710a-434e-9d06-9728fd902047"))
meu.saveDF()

========================= masterManualEntries(install=False) =========================
Current Time is Thu Dec 16, 2021 20:12 for Getting Manual Entries Data From Main Pickle File
Process [Getting Manual Entries Data From Main Pickle File] Took 0.7 Seconds
========================= MasterDB Data(dtype=Search, dbs=None) =========================
  ==> Added New Row [ArtistName                          Cowboy & Spin Girl
Discogs                                         632402
MusicBrainz    258815998727572057361637915645274805163
AllMusic                                    0000101460
Name: a4831407-7709-48b2-8424-2ccab76cc111, dtype: object]
  ==> Added New Row [ArtistName                                 Coyabalites
Discogs                                        1511252
MusicBrainz    153363932161423635557855629209525501336
Name: d42620fd-16e6-4d12-b93e-852e09c99cd7, dtype: object]
  ==> Added New Row [ArtistName                                           DFL
Discogs                       

In [9]:
meu = masterManualEntriesUtils()
meu.newArtist("Gary Celebrity",  Discogs="1722835",  MusicBrainz=meu.getmbid("https://musicbrainz.org/artist/47ae487b-8674-4e31-9374-000009074895"))
meu.newArtist("Gerry Milnes",  Discogs="2511267",  MusicBrainz=meu.getmbid("https://musicbrainz.org/artist/a8fe88ab-2a54-4bd9-b9ad-b6e9529cdc03"))
meu.newArtist("Geto Boys",  Discogs="154551",  MusicBrainz=meu.getmbid("https://musicbrainz.org/artist/a78ca453-3a53-4b19-8b2d-61b10dd45558"), AllMusic="0000077283", RateYourMusic="2561", Deezer="2828")
meu.newArtist("Go Go Market",  Discogs="1544560",  MusicBrainz=meu.getmbid("https://musicbrainz.org/artist/b03855b1-6981-4d1f-b028-7cb17e1958c1"))
meu.newArtist("Go National",  Discogs="1409411",  MusicBrainz=meu.getmbid("https://musicbrainz.org/artist/aa7fa8b6-4117-4dc0-938d-f56b95b043b9"))
meu.newArtist("Harry Anselmi",  Discogs="2827120",  AllMusic="0001377169")
meu.newArtist("Henri Landry",  Discogs="3333719",  MusicBrainz=meu.getmbid("https://musicbrainz.org/artist/f62c62e9-c852-4c88-9761-d7ba7438b228"))
meu.newArtist("Homethrust",  Discogs="1222476",  MusicBrainz=meu.getmbid("https://musicbrainz.org/artist/535f1678-5720-45fd-9a45-578b924ad5ba"))
meu.newArtist("Hopscotch Army",  Discogs="1611545",  MusicBrainz=meu.getmbid("https://musicbrainz.org/artist/9c0dcef2-e624-460c-940d-47487ea38f16"))
meu.newArtist("Intro To India",  Discogs="378401",  MusicBrainz=meu.getmbid("https://musicbrainz.org/artist/ab7b733d-11ba-49f1-96a7-12c845c0ea38"))
meu.newArtist("J.C. Superska",  Discogs="2677656",  MusicBrainz=meu.getmbid("https://musicbrainz.org/artist/1cec4517-213c-4584-b2f2-8aed9cfb876d"))
meu.newArtist("Jetpack",  Discogs="1999282",  AllMusic="0003274641")
meu.newArtist("Joaquina",  Discogs="1608893",  MusicBrainz=meu.getmbid("https://musicbrainz.org/artist/884bbe66-115b-4127-8a5a-892a9db99799"))
meu.newArtist("Joe Strummer & The Mescaleros",  Discogs="272595",  MusicBrainz=meu.getmbid("https://musicbrainz.org/artist/39c1e474-647e-42ef-a157-fcfb30c2c2ff"), AllMusic="0000806411", RateYourMusic="1618")
meu.newArtist("John Verity Band",  Discogs="1501877",  MusicBrainz=meu.getmbid("https://musicbrainz.org/artist/ea66defa-d24d-438c-b5c3-1508a7880381"))
meu.newArtist("Jonny Chan And The New Dynasty 6",  Discogs="2045262",  MusicBrainz=meu.getmbid("https://musicbrainz.org/artist/53e2e840-3a21-4f16-8f41-b314214bf779"))
meu.newArtist("Jr. Boss",  Discogs="3889349",  MusicBrainz=meu.getmbid("https://musicbrainz.org/artist/8f8f5031-1c22-47f6-ac71-7769141907fb"))
meu.newArtist("June & The Exit Wounds",  Discogs="893556",  MusicBrainz=meu.getmbid("https://musicbrainz.org/artist/3534a50d-9c98-468e-bf28-3aae1d7546dd"))
meu.newArtist("Karmic Jera",  Discogs="1906176",  MusicBrainz=meu.getmbid("https://musicbrainz.org/artist/a8020265-0a90-4ea3-b754-c6385f8a2ec2"))
meu.saveDF()

========================= masterManualEntries(install=False) =========================
Current Time is Fri Dec 17, 2021 19:55 for Getting Manual Entries Data From Main Pickle File
Process [Getting Manual Entries Data From Main Pickle File] Took 0.7 Seconds
========================= MasterDB Data(dtype=Search, dbs=None) =========================
  ==> Added New Row [ArtistName                             Gary Celebrity
Discogs                                       1722835
MusicBrainz    23170537431468541349045987211133250571
Name: 2fda608a-905d-449e-a167-1aba1dfed723, dtype: object]
  ==> Added New Row [ArtistName                               Gerry Milnes
Discogs                                       2511267
MusicBrainz    30580385087989407988740566325226577329
Name: cc308b86-4920-4536-88d6-446a108df67d, dtype: object]
  ==> Added New Row [ArtistName                                     Geto Boys
Discogs                                           154551
MusicBrainz      28959298203977014

In [ ]:

meu.newArtist("Katie's Dimples",  Discogs="",  MusicBrainz=meu.getmbid(""))
meu.newArtist("Ken Kunin",  Discogs="",  MusicBrainz=meu.getmbid(""))
meu.newArtist("Kenny Wayne Shepherd Band",  Discogs="",  MusicBrainz=meu.getmbid(""))
meu.newArtist("Kerry Kearney",  Discogs="",  MusicBrainz=meu.getmbid(""))
meu.newArtist("LBC Movement",  Discogs="",  MusicBrainz=meu.getmbid(""))
meu.newArtist("La Lengua Asesina",  Discogs="",  MusicBrainz=meu.getmbid(""))
meu.newArtist("Larry Perkins",  Discogs="",  MusicBrainz=meu.getmbid(""))
meu.newArtist("Laughing Sky",  Discogs="",  MusicBrainz=meu.getmbid(""))
meu.newArtist("Lodestone",  Discogs="",  MusicBrainz=meu.getmbid(""))
meu.newArtist("Lovelight Shine",  Discogs="",  MusicBrainz=meu.getmbid(""))
meu.newArtist("Lower 48",  Discogs="",  MusicBrainz=meu.getmbid(""))
meu.newArtist("Luck Of Aleia",  Discogs="",  MusicBrainz=meu.getmbid(""))
meu.newArtist("LuckyMe",  Discogs="",  MusicBrainz=meu.getmbid(""))
meu.newArtist("Lungbrush",  Discogs="",  MusicBrainz=meu.getmbid(""))
meu.newArtist("M.A.R.R.S",  Discogs="",  MusicBrainz=meu.getmbid(""))
meu.newArtist("Mama Roo",  Discogs="",  MusicBrainz=meu.getmbid(""))
meu.newArtist("Marvelkind",  Discogs="",  MusicBrainz=meu.getmbid(""))
meu.newArtist("Marvy Darling",  Discogs="",  MusicBrainz=meu.getmbid(""))
meu.newArtist("Matt Marque",  Discogs="",  MusicBrainz=meu.getmbid(""))
meu.newArtist("Mega Super Ultra",  Discogs="",  MusicBrainz=meu.getmbid(""))
meu.newArtist("Michael Franti & Spearhead",  Discogs="",  MusicBrainz=meu.getmbid(""))
meu.newArtist("Miighty Flashlight",  Discogs="",  MusicBrainz=meu.getmbid(""))
meu.newArtist("Mitch Harrell",  Discogs="",  MusicBrainz=meu.getmbid(""))
meu.newArtist("Monaural",  Discogs="",  MusicBrainz=meu.getmbid(""))
meu.newArtist("Morgan Harris",  Discogs="",  MusicBrainz=meu.getmbid(""))
meu.newArtist("Morphine Angel",  Discogs="",  MusicBrainz=meu.getmbid(""))
meu.newArtist("Mrs. Hippie",  Discogs="",  MusicBrainz=meu.getmbid(""))
meu.newArtist("Nancy Moore",  Discogs="",  MusicBrainz=meu.getmbid(""))
meu.newArtist("New Enemies",  Discogs="",  MusicBrainz=meu.getmbid(""))
meu.newArtist("Nick Clemons",  Discogs="",  MusicBrainz=meu.getmbid(""))
meu.newArtist("Nordic Union",  Discogs="",  MusicBrainz=meu.getmbid(""))
meu.newArtist("Outcry",  Discogs="",  MusicBrainz=meu.getmbid(""))
meu.newArtist("Owltian Mia",  Discogs="",  MusicBrainz=meu.getmbid(""))
meu.newArtist("Pacifier",  Discogs="",  MusicBrainz=meu.getmbid(""))
meu.newArtist("Pat & Lolly Vegas",  Discogs="",  MusicBrainz=meu.getmbid(""))
meu.newArtist("Pengshui",  Discogs="",  MusicBrainz=meu.getmbid(""))
meu.newArtist("Pequod",  Discogs="",  MusicBrainz=meu.getmbid(""))
meu.newArtist("Perfect World",  Discogs="",  MusicBrainz=meu.getmbid(""))
meu.newArtist("Pete Benz",  Discogs="",  MusicBrainz=meu.getmbid(""))
meu.newArtist("Peter Zaremba's Love Delegation",  Discogs="",  MusicBrainz=meu.getmbid(""))
meu.newArtist("Precious Paris",  Discogs="",  MusicBrainz=meu.getmbid(""))
meu.newArtist("Prevent Falls",  Discogs="",  MusicBrainz=meu.getmbid(""))
meu.newArtist("Puff Daddy & The Family",  Discogs="",  MusicBrainz=meu.getmbid(""))
meu.newArtist("Pure 13",  Discogs="",  MusicBrainz=meu.getmbid(""))
meu.newArtist("Radar Mercury",  Discogs="",  MusicBrainz=meu.getmbid(""))
meu.newArtist("Reality Jones",  Discogs="",  MusicBrainz=meu.getmbid(""))
meu.newArtist("Ricky Skaggs & Kentucky Thunder",  Discogs="",  MusicBrainz=meu.getmbid(""))
meu.newArtist("Roky Erickson And The Aliens",  Discogs="",  MusicBrainz=meu.getmbid(""))
meu.newArtist("Ronnie & The Pomona Casuals",  Discogs="",  MusicBrainz=meu.getmbid(""))
meu.newArtist("Rotten Apples",  Discogs="",  MusicBrainz=meu.getmbid(""))
meu.newArtist("Roy Loney & the Phantom Movers",  Discogs="",  MusicBrainz=meu.getmbid(""))
meu.newArtist("Royce Rizzy",  Discogs="",  MusicBrainz=meu.getmbid(""))
meu.newArtist("Runforyerlife",  Discogs="",  MusicBrainz=meu.getmbid(""))
meu.newArtist("Sam Cooke & The Soul Stirrers",  Discogs="",  MusicBrainz=meu.getmbid(""))
meu.newArtist("Sayitainttone",  Discogs="",  MusicBrainz=meu.getmbid(""))
meu.newArtist("Scott Meldrum",  Discogs="",  MusicBrainz=meu.getmbid(""))
meu.newArtist("Scott Ritcher",  Discogs="",  MusicBrainz=meu.getmbid(""))
meu.newArtist("Sea Tiger",  Discogs="",  MusicBrainz=meu.getmbid(""))
meu.newArtist("Short Round",  Discogs="",  MusicBrainz=meu.getmbid(""))
meu.newArtist("Shuttlecock",  Discogs="",  MusicBrainz=meu.getmbid(""))
meu.newArtist("Silver Watermelon",  Discogs="",  MusicBrainz=meu.getmbid(""))
meu.newArtist("Sir Bald Diddley and his Wig Outs",  Discogs="",  MusicBrainz=meu.getmbid(""))
meu.newArtist("Skiptracer",  Discogs="",  MusicBrainz=meu.getmbid(""))
meu.newArtist("Some Day Soon",  Discogs="",  MusicBrainz=meu.getmbid(""))
meu.newArtist("Songs For Emma",  Discogs="",  MusicBrainz=meu.getmbid(""))
meu.newArtist("Sonny & Cher",  Discogs="",  MusicBrainz=meu.getmbid(""))
meu.newArtist("Southern Style Dj's",  Discogs="",  MusicBrainz=meu.getmbid(""))
meu.newArtist("Special Skank",  Discogs="",  MusicBrainz=meu.getmbid(""))
meu.newArtist("Squeaky Burger",  Discogs="",  MusicBrainz=meu.getmbid(""))
meu.newArtist("Stackabones",  Discogs="",  MusicBrainz=meu.getmbid(""))
meu.newArtist("Standpoint",  Discogs="",  MusicBrainz=meu.getmbid(""))
meu.newArtist("Steady Earnest",  Discogs="",  MusicBrainz=meu.getmbid(""))
meu.newArtist("Stephen Malkmus & The Jicks",  Discogs="",  MusicBrainz=meu.getmbid(""))
meu.newArtist("Steve Lawrence & Eydie Gorme",  Discogs="",  MusicBrainz=meu.getmbid(""))
meu.newArtist("Steve Margoshes",  Discogs="",  MusicBrainz=meu.getmbid(""))
meu.newArtist("Stressball",  Discogs="",  MusicBrainz=meu.getmbid(""))
meu.newArtist("StylipS",  Discogs="",  MusicBrainz=meu.getmbid(""))
meu.newArtist("Supergrub",  Discogs="",  MusicBrainz=meu.getmbid(""))
meu.newArtist("Tee And Thee Crumpets",  Discogs="",  MusicBrainz=meu.getmbid(""))
meu.newArtist("Telephone Man",  Discogs="",  MusicBrainz=meu.getmbid(""))
meu.newArtist("Tex Beaumont",  Discogs="",  MusicBrainz=meu.getmbid(""))
meu.newArtist("The Blood Drained Cows",  Discogs="",  MusicBrainz=meu.getmbid(""))
meu.newArtist("The Blues Brothers Band",  Discogs="",  MusicBrainz=meu.getmbid(""))
meu.newArtist("The Concussion Ensemble",  Discogs="",  MusicBrainz=meu.getmbid(""))
meu.newArtist("The Cool Jerks",  Discogs="",  MusicBrainz=meu.getmbid(""))
meu.newArtist("The Diane Linkletter Experience",  Discogs="",  MusicBrainz=meu.getmbid(""))
meu.newArtist("The Excessories",  Discogs="",  MusicBrainz=meu.getmbid(""))
meu.newArtist("The Fisticuffs Bluff",  Discogs="",  MusicBrainz=meu.getmbid(""))
meu.newArtist("The Fresh Beat Band",  Discogs="",  MusicBrainz=meu.getmbid(""))
meu.newArtist("The Great Brain",  Discogs="",  MusicBrainz=meu.getmbid(""))
meu.newArtist("The Halo Bit",  Discogs="",  MusicBrainz=meu.getmbid(""))
meu.newArtist("The Higher Burning Fire",  Discogs="",  MusicBrainz=meu.getmbid(""))
meu.newArtist("The Inspector",  Discogs="",  MusicBrainz=meu.getmbid(""))
meu.newArtist("The Jimi Hendrix Experience",  Discogs="",  MusicBrainz=meu.getmbid(""))
meu.newArtist("The Johnny Burnette Trio",  Discogs="",  MusicBrainz=meu.getmbid(""))
meu.newArtist("The Kickovers",  Discogs="",  MusicBrainz=meu.getmbid(""))
meu.newArtist("The Lofty Pillars",  Discogs="",  MusicBrainz=meu.getmbid(""))
meu.newArtist("The Lookyloos",  Discogs="",  MusicBrainz=meu.getmbid(""))
meu.newArtist("The McNeely-Levin-Skinner Band",  Discogs="",  MusicBrainz=meu.getmbid(""))
meu.newArtist("The Now Time Delegation",  Discogs="",  MusicBrainz=meu.getmbid(""))
meu.newArtist("The Okra All-Stars",  Discogs="",  MusicBrainz=meu.getmbid(""))
meu.newArtist("The Part-Time Losers",  Discogs="",  MusicBrainz=meu.getmbid(""))
meu.newArtist("The Penthouse 5",  Discogs="",  MusicBrainz=meu.getmbid(""))
meu.newArtist("The Princeton Footnotes",  Discogs="",  MusicBrainz=meu.getmbid(""))
meu.newArtist("The Shimmer Kids",  Discogs="",  MusicBrainz=meu.getmbid(""))
meu.newArtist("The Smokejumpers",  Discogs="",  MusicBrainz=meu.getmbid(""))
meu.newArtist("The Swaggerts",  Discogs="",  MusicBrainz=meu.getmbid(""))
meu.newArtist("The Tennessee Tearjerkers",  Discogs="",  MusicBrainz=meu.getmbid(""))
meu.newArtist("The Uphill Gardeners",  Discogs="",  MusicBrainz=meu.getmbid(""))
meu.newArtist("The Weak Moments",  Discogs="",  MusicBrainz=meu.getmbid(""))
meu.newArtist("The Wharton Tiers Ensemble",  Discogs="",  MusicBrainz=meu.getmbid(""))
meu.newArtist("They Come In Threes",  Discogs="",  MusicBrainz=meu.getmbid(""))
meu.newArtist("Thinking Man",  Discogs="",  MusicBrainz=meu.getmbid(""))
meu.newArtist("Third Age",  Discogs="",  MusicBrainz=meu.getmbid(""))
meu.newArtist("Thrillcat",  Discogs="",  MusicBrainz=meu.getmbid(""))
meu.newArtist("Tokino Sora",  Discogs="",  MusicBrainz=meu.getmbid(""))
meu.newArtist("Tony Scott & Horst Jankowski Trio",  Discogs="",  MusicBrainz=meu.getmbid(""))
meu.newArtist("Totally Travis",  Discogs="",  MusicBrainz=meu.getmbid(""))
meu.newArtist("Travis Pickle",  Discogs="",  MusicBrainz=meu.getmbid(""))
meu.newArtist("Tribal Lust And The Horny Natives",  Discogs="",  MusicBrainz=meu.getmbid(""))
meu.newArtist("Tyler Keith",  Discogs="",  MusicBrainz=meu.getmbid(""))
meu.newArtist("Vegas Thunder",  Discogs="",  MusicBrainz=meu.getmbid(""))
meu.newArtist("Velour Motel",  Discogs="",  MusicBrainz=meu.getmbid(""))
meu.newArtist("Virginia Sil'hooettes",  Discogs="",  MusicBrainz=meu.getmbid(""))
meu.newArtist("WE55",  Discogs="",  MusicBrainz=meu.getmbid(""))
meu.newArtist("Wave Workers Foundation",  Discogs="",  MusicBrainz=meu.getmbid(""))
meu.newArtist("Wayne Henderson And The Next Crusade",  Discogs="",  MusicBrainz=meu.getmbid(""))
meu.newArtist("West, Space and Love",  Discogs="",  MusicBrainz=meu.getmbid(""))
meu.newArtist("Witch Hazel Sound",  Discogs="",  MusicBrainz=meu.getmbid(""))
meu.newArtist("Written in the Sand",  Discogs="",  MusicBrainz=meu.getmbid(""))
meu.newArtist("Xmarsx",  Discogs="",  MusicBrainz=meu.getmbid(""))
meu.newArtist("Youngsbower",  Discogs="",  MusicBrainz=meu.getmbid(""))
meu.newArtist("Сборник",  Discogs="",  MusicBrainz=meu.getmbid(""))

In [ ]:
meu.getMMEByArtist("69 Tribe", 5)

In [ ]:
meu.newArtist("69 Tribe", Discogs="1442812", RateYourMusic="369095")

In [ ]:
meu.saveDF()

In [ ]:
for dbToUseID,dbToUseAlbumMatchData in matchedArtistData.items():
    dbToUseResults[dbToUseID] = {}
    for db,dbAlbumMatchData in dbToUseAlbumMatchData.items():

In [ ]:
io = fileIO()
artistSummaryDF     = io.get("music_artistSummaryDF.p")
myMusicArtistAlbums = DataFrame({artistName: {"ArtistName": artistName, "Albums": Series(df["AlbumName"].to_list())} for artistName,df in artistSummaryDF.groupby("ArtistName")}).T

In [ ]:
myMusicMasterIDs = {}

N = myMusicArtistAlbums.shape[0]
ts = timestat("Matching {0} Artists From My Music".format(N))
matmdb = matchArtistToMasterDB(masterArtistAlbumsData)
for n,(idx,row) in enumerate(myMusicArtistAlbums.iterrows()):
    matmdb.setArtistInfo(row)
    matmdb.findArtistMatches()
    matmdb.findArtistAlbumMatches()
    
    myMusicMasterIDs[idx] = matmdb.getAlbumMatchResults()
    if len(myMusicMasterIDs) > 100:
        ts.update(n=n+1, N=N)
        break
ts.stop()

In [ ]:
matchedArtistData      = {item["ArtistID"]: item["Results"] for item in }
#matchedArtistAlbumData = {item["ArtistID"]: item["Results"] for item in mmg.matchedArtistAlbumData}
#dbToUseResults         = getAlbumMatchResults(matchedArtistAlbumData)


In [ ]:
def getAlbumMatchResults(matchedArtistAlbumData):
    dbToUseResults = {}
    for dbToUseID,dbToUseAlbumMatchData in matchedArtistAlbumData.items():
        dbToUseResults[dbToUseID] = {}
        for db,dbAlbumMatchData in dbToUseAlbumMatchData.items():
            if dbAlbumMatchData is None:
                continue
            albumResult = DataFrame(dbAlbumMatchData).T
            dbIDMatch   = albumResult[(albumResult["Match"] >= 2.0) & (albumResult["Exact"] >= 1.0)].sort_values(by=["Match","Score"], ascending=False).head(1)
            if dbIDMatch.shape[0] > 0:
                dbToUseResults[dbToUseID][db] = dbIDMatch.index[0]
    dbToUseResults = Series(dbToUseResults).apply(Series)
    return dbToUseResults

In [ ]:
def getBestMatch(matchResult):
    value  = matchResult[matchResult["Match"] >= 2.0]
    retval = None if value.shape[0] == 0 else list(value.head(1).index)[0]
    return retval

Series(myMusicMasterIDs).apply(getBestMatch)

In [ ]:
from matchArtistToDB import masterMatch

class matchArtistToMasterDB:
    def __init__(self, masterArtistAlbumsData, debug=False, detail=0):
        self.mdbData = masterArtistAlbumsData
        self.mdbDataSearchArtistNames = masterArtistAlbumsData["ArtistName"].str.upper()
        
        self.debug  = debug
        self.detail = detail
        
        self.matchNumArtistName     = None
        self.matchArtistNameCutoff  = None
        self.matchArtistAlbumCutoff = None
        self.matchNumArtistAlbums   = None
        self.matchScore             = None

        self.artistData             = None
        self.artistSearchResults    = {}
        self.albumSearchResults     = {}
        
        self.setThresholds()
        
        self.mm = masterMatch()
        
        
    #############################################################################################################################
    # Search Inputs
    #############################################################################################################################
    def setArtistInfo(self, artistData):
        self.artistData      = artistData
        self.artistName      = artistData["ArtistName"]
        self.artistID        = artistData.name
        self.artistAlbums    = artistData["Albums"][artistData["Albums"].notna()] if artistData.get("Albums") is not None else []
        self.artistNumAlbums = len(self.artistAlbums)
        
        
    #############################################################################################################################
    # I/O
    #############################################################################################################################
    def getArtistMatchResults(self):
        matchResults = {}
        if isinstance(self.artistSearchResults, DataFrame) and self.artistSearchResults.shape[0] > 0:
            dbMatchResult = self.artistSearchResults.sort_values(by="Levenshtein", ascending=False)
            matchResults  = dbMatchResult.T.to_dict()
        return {"ArtistID": self.artistID, "ArtistName": self.artistName, "Results": matchResults}
    
    
    def getAlbumMatchResults(self):
        matchResults = {}
        if isinstance(self.albumSearchResults, DataFrame) and self.albumSearchResults.shape[0] > 0:
            dbMatchResult = self.albumSearchResults.sort_values(by=["Match", "Score"], ascending=False)
            dbMatchResult["ArtistName"] = dbMatchResult.apply(lambda row: self.mdbData.loc[row.name, "ArtistName"], axis=1)
            matchResults = dbMatchResult.T.to_dict()
        return {"ArtistID": self.artistID, "ArtistName": self.artistName, "ArtistNumAlbums": self.artistNumAlbums, "Results": matchResults}
            
        
    def setThresholds(self, **kwargs):
        self.matchNumArtistName     = kwargs.get("numArtists", 2)
        self.matchArtistNameCutoff  = kwargs.get("artistCutoff", 0.8)
        self.matchNumArtistAlbums   = kwargs.get("numAlbums", 2)
        self.matchArtistAlbumCutoff = kwargs.get("albumCutoff", 0.9)
        self.matchScore             = kwargs.get("score", 1.8)
        
        
    #############################################################################################################################
    # Search Functions For Artists
    #############################################################################################################################
    def findArtistMatches(self):
        #searchArtists = self.mdbData.getDBSearchArtistNames(db)
        searchArtists = self.mdbDataSearchArtistNames
        
        ####### Only Look At Names Within Range Of Artist
        tmp = searchArtists.apply(len)
        artistNameUpper = self.artistName.upper()
        lenArtistName   = len(artistNameUpper)
        
        idxs = abs(tmp - lenArtistName) < 5
        searchArtists = searchArtists.loc[idxs]
        #searchResults = self.mm.matchNames(tomatch=searchResults, value=artistNameUpper)
        searchResults = searchArtists.apply(self.mm.getLevenshtein, x2=artistNameUpper)
        searchResults.name = "Levenshtein"
        idxs   = searchResults >= self.matchArtistNameCutoff
        result = DataFrame(searchArtists[idxs]).join(searchResults[idxs])
        self.artistSearchResults = result
        

    #############################################################################################################################
    # Search Functions For Artists + Albums
    #############################################################################################################################
    def findArtistAlbumMatches(self):
        ### 2nd: Score Each Match Using Albums
        idxResults = {}
        for idx,artistMatchData in self.artistSearchResults.iterrows():
            artistMatchAlbums = [str(x).upper() for x in self.mdbData.loc[idx, "Albums"] if x is not None]
            numAlbums         = len(artistMatchAlbums)
            idxResults[idx]   = {"NumAlbums": numAlbums}
            idxResults[idx]["Match"]  = -1.0
            idxResults[idx]["Exact"]  = -1.0
            idxResults[idx]["Score"]  = -1.0
            idxResults[idx]["M"]      = 0
            idxResults[idx]["Albums"] = artistMatchAlbums if self.detail >= 2 else 0
            if numAlbums > 0:
                try:
                    M    = self.artistAlbums.apply(lambda artistAlbum: Series([self.mm.getLevenshtein(matchAlbum, artistAlbum.upper()) for matchAlbum in artistMatchAlbums]))
                except:
                    print("ERROR Producing M matrix")
                    print("  MatchAlbums:",artistMatchAlbums)
                    print(" ArtistAlbums:",self.artistAlbums)
                    continue
                    
                try:
                    Max0 = M.max(axis=0)
                    Max1 = M.max(axis=1)
                except:
                    print("ERROR Producing M.max() matrix")
                    print("  MatchAlbums:",artistMatchAlbums)
                    print(" ArtistAlbums:",self.artistAlbums)
                    idxResults[idx]["M"]      = M if self.detail >= 1 else 0
                    continue

                try:
                    idxResults[idx]["Match"]  = min([(Max0 >= 0.825).sum(), (Max1 >= 0.825).sum()])
                    idxResults[idx]["Exact"]  = min([(Max0 >= 0.875).sum(), (Max1 >= 0.875).sum()])
                    idxResults[idx]["Score"]  = min([Max0[Max0 > self.matchArtistAlbumCutoff].sum(), Max1[Max1 > self.matchArtistAlbumCutoff].sum()])
                    idxResults[idx]["M"]      = M if self.detail >= 1 else 0
                    idxResults[idx]["Albums"] = artistMatchAlbums if self.detail >= 2 else 0
                except:
                    print("ERROR Producing Match/Exact/Score")
                    print("  MatchAlbums:",artistMatchAlbums)
                    print(" ArtistAlbums:",self.artistAlbums)
                    idxResults[idx]["M"]      = M if self.detail >= 1 else 0
                    continue
            idxResults[idx] = Series(idxResults[idx]).fillna(0)
        result = Series(idxResults).apply(Series) if len(idxResults) > 0 else DataFrame()
        #ts.stop()
        self.albumSearchResults = result

In [ ]:
len(myMusicMatchIDs)

In [ ]:
len(noMusicMatchs)

In [ ]:
multiMusicMatchs['AFI']

# Music Details

In [ ]:
from os import walk
import subprocess

def getPathSize(path,retUnit="M"):
    units = {"B": pow(1024,0), "K": pow(1024,1), "M": pow(1024,2), "G": pow(1024,3)}

    try:
        result = subprocess.check_output(['du','-sh', path]).split()[0].decode('utf-8')
    except:
        return None
    size   = float(result[:-1])
    unit   = result[-1]
    size  *= (units[unit])/(units[retUnit])
    return size


def getPathFiles(path):
    pathFiles = []
    pathDirs  = {}
    for i,(root, dirs, files) in enumerate(walk(path)):
        pathFiles += [dirUtil(root).join(ifile) for ifile in files]
        if len(dirs) > 0:
            pathDirs.update({root: dirs})
    return {"Files": pathFiles, "Dirs": pathDirs}


def getPathTimestamp(path):
    return fsPath(path).modified


def getPathSummary(albumPathData):
    nSubdirs = len(albumPathData['Dirs'])
    nFiles   = len(albumPathData["Files"])
    ftypes   = Series([fsInfo(ifile).path.suffix for ifile in albumPathData["Files"]]).value_counts().to_dict()
    return {"nSubdirs": nSubdirs, "nFiles": nFiles, "ftypes": ftypes}

In [ ]:
def getArtistSummaryData(artistAlbums, albumsSummaryData):
    artistSummaryData = {}
    for artistName,artistData in artistAlbums.items():
        artistSummaryData[artistName] = {}
        for albumType,albumTypeData in artistData.items():
            artistSummaryData[artistName][albumType] = {album: albumsSummaryData.loc[album] for album in albumTypeData}

    retval = {}
    for artistName,artistData in artistSummaryData.items():
        for albumType,albumTypeData in artistData.items():
            for album,albumData in albumTypeData.items():
                albumName = dirUtil(album).name.split(" :: ")[0]
                key   = (artistName,albumType,albumName)
                value = albumData
                if retval.get(key) is not None:
                    print(key)
                retval[key] = value

    df = DataFrame(retval).T
    df = df.reset_index().rename({'level_0': "ArtistName", 'level_1': "AlbumType", 'level_2': "AlbumName"}, axis=1)
    return df


def getSummaryDataRollup(x):
    colname = x.name
    if colname in ["Size", "nFiles", "nSubdirs"]:
        return {colname: x.astype(int).sum()}
    elif colname in ["Timestamp"]:
        return {colname: x.max()}
    elif colname in ["AlbumName", "AlbumType", "ArtistName"]:
        nunique = x.nunique()
        if nunique == 1:
            return {colname: x.unique()[0]}
        else:
            return {"n{0}s".format(colname[:-4]): nunique}
    elif colname in ["ftypes"]:
        return {colname: x.apply(Series).sum().to_dict()}
    else:
        raise ValueError("Not sure what to do with column [{0}]".format(colname))
    return 0


def getSummaryRollupDataFrame(df, byArtist=True):
    retval = []
    keys   = ["ArtistName","AlbumType","AlbumName"]
    gby    = keys[:1] if byArtist else keys[:2]
    ignore = keys[-2:] if byArtist else keys[-1:]
    for idx,idxDF in df.groupby(gby):
        result = {}
        for item in idxDF.apply(summary).values:
            if sum([x in item.keys() for x in ignore]) == 0:
                result.update(item)    
        retval.append(result)
        
    summaryDF = DataFrame(retval).fillna(0)
    summaryDF['nAlbums'] = summaryDF['nAlbums'].astype(int)
    return summaryDF

In [ ]:
from pandas import DataFrame, Series
from listUtils import getFlatList

tsAll = timestat("Collecting Artist + Album Details Information")

ts = timestat("Flattening Albums")
artistAlbums = {artistName: {albumType: albumTypeData.albums for albumType,albumTypeData in artistData.albumData.items()} for artistName,artistData in mmpdData.items()}
albumsList   = getFlatList([y for y in getFlatList([x.values() for x in artistAlbums.values()]) if len(y) > 0])
ts.stop()

ts = timestat("Getting Size For {0} Albums".format(len(albumsList)))
albumSizes = {album: getPathSize(album) for album in albumsList}
ts.stop()

ts = timestat("Getting Files/Subdirs For {0} Albums".format(len(albumsList)))
albumFiles = {album: getPathFiles(album) for album in albumsList}
ts.stop()

ts = timestat("Getting Timestamps For {0} Albums".format(len(albumsList)))
albumTimestamps = {album: getPathTimestamp(album) for album in albumsList}
ts.stop()

ts = timestat("Getting Summary For {0} Albums".format(len(albumsList)))
albumSummaries = {album: getPathSummary(albumData) for album,albumData in albumFiles.items()}
ts.stop()

ts = timestat("Joining Data For {0} Albums".format(len(albumsList)))
albumsSummaryData = DataFrame(albumSummaries).T
albumsSummaryData = albumsSummaryData.join(Series(albumTimestamps, name="Timestamp"))
albumsSummaryData = albumsSummaryData.join(Series(albumSizes, name="Size"))
ts.stop()

ts = timestat("Creating Summary DataFrame")
artistSummaryDF = getArtistSummaryData(artistAlbums, albumsSummaryData)
ts.stop()

tsAll.stop()

In [ ]:
io = fileIO()
io.save(idata=artistSummaryDF, ifile="music_artistSummaryDF.p")
io.save(idata=albumsSummaryData, ifile="music_albumsSummaryData.p")
io.save(idata=albumSummaries, ifile="music_albumSummaries.p")

In [ ]:
artistRollupDF = getSummaryRollupDataFrame(artistSummaryDF, byArtist=True)

In [ ]:
"""
('A Flock Of Seagulls', 'Match', 'Greatest Hits Remixed')
('A Flock Of Seagulls', 'Match', 'I Ran')
('A Flock Of Seagulls', 'Match', 'Magic')
('A Global Threat', 'Match', 'Here We Are')
('A Tribe Called Quest', 'Match', 'Midnight Marauders')
('A Tribe Called Quest', 'Match', 'The Love Movement')
('A Tribe Called Quest', 'Match', 'The Low End Theory')
('A Tribe Called Quest', 'Match', 'The Lost Tribes')
('ABC', 'Match', 'Alphabet City')
('ABC', 'Match', 'Skyscraping')
('AFI', 'Match', 'A Fire Inside EP')
('Abdullah', 'Match', 'Graveyard Poetry')
('Abdullah', 'Match', 'Abdullah')
('Abominator', 'Match', "Damnation's Prophecy")
('Above The Law', 'Match', 'Call It What U Want')
('Abscess', 'Match', 'Tormented')
('Abscess', 'Match', 'Urine Junkies')
('Absolution', 'Match', 'Complete Discography')
('Action League', 'Match', 'Interrupt This Program')
('Adam Ant', 'Match', 'Antics In The Forbidden Zone')
('Addrisi Brothers', 'Match', 'Cherrystone')
('Adele', 'Match', '25')
('Adele', 'Match', 'Chasing Pavements')
('Adele', 'Match', 'Cold Shoulder')
('Adele', 'Match', 'Make You Feel My Love')
('Adele', 'Match', 'Rolling In The Deep')
('Adele', 'Match', 'Rumour Has It')
('Adele', 'Match', 'Skyfall')
('Adele', 'Match', 'Someone Like You')
('Adele', 'Match', 'Turning Tables')
('Adele', 'Match', 'iTunes Festival: London 2011')
('Adele', 'Match', 'iTunes Live From SoHo')
('Adele', 'Match', 'Hometown Glory')
('Aerosmith', 'Match', "Aerosmith's Greatest Hits")
('Agent 99', 'Match', 'Little Pieces: 1993-1995')
('Agnostic Front', 'Match', 'Riot, Riot, Upstart')
('Air Supply', 'Match', 'Greatest Hits Live... Now And Forever')
('Air Supply', 'Match', 'The Definitive Collection')
('Akon', 'Match', 'Freedom')
('Al Green', 'Match', 'Anthology')
('Al Green', 'Match', 'Call Me')
('Alan Jackson', 'Match', 'Everything I Love')
('Alannah Myles', 'Match', 'A-Lan-Nah')
('Alannah Myles', 'Match', 'Arival')
('Alannah Myles', 'Match', 'Alannah Myles')
('Alannah Myles', 'Match', 'Black Velvet')
('Albert Hammond', 'Match', 'Albert Hammond')
('Alex Britti', 'Match', 'It.pop')
('Alice In Chains', 'Match', 'Facelift')
('Alice In Chains', 'Match', 'Live')
('Alice In Chains', 'Match', 'The Devil Put Dinosaurs Here')
('Alice In Chains', 'Match', 'Bank Heist')
('Alice In Chains', 'Match', 'Music Bank')
('Alice In Chains', 'Match', 'Nothing Safe: The Best Of The Box')
('Alice In Chains', 'Match', 'The Music Of Alice In Chains')
('Alice In Chains', 'Match', 'Sap')
('Alice In Chains', 'Match', 'Dirt')
('All Out War', 'Match', 'For Those Who Were Crucified')
('All-4-One', 'Match', 'I Can Love You Like That')
('Alley Boy', 'Match', 'Nigganati')
('Aloha', 'Match', 'Sugar')
('Aloha', 'Match', 'Aloha')
('Alphaville', 'Match', 'Salvation')
('Amanda Marshall', 'Match', 'Amanda Marshall')
('Amber Asylum', 'Match', 'Songs Of Sex And Death')
('Amber Asylum', 'Match', 'The Supernatural Parlour Collection')
('Amel Larrieux', 'Match', 'Infinite Possibilities')
('America', 'Match', 'Greatest Hits')
('Amoeba', 'Match', 'Pivot')
('Amoeba', 'Match', 'Watchful')
('Amorphis', 'Match', 'Am Universum')
('Amorphis', 'Match', 'Elegy')
('Amorphis', 'Match', 'The Karelian Isthmus')
('Amorphis', 'Match', 'Tuonela')
('Amorphis', 'Match', 'Black Winter Day')
('Amorphis', 'Match', 'My Kantele')
('Amorphis', 'Match', 'Privilege Of Evil')
('Amsvartner', 'Match', 'Dreams')
('Amy Grant', 'Match', 'In Concert')
('Amy Grant', 'Match', 'In Motion (The Remixes)')
('Amy Grant', 'Match', "My Father's Eyes")
('Amy Grant', 'Match', 'Unguarded')
('Amy Grant', 'Match', 'My Best Christmas')
('Amy Grant', 'Match', 'The Collection')
('Amy Grant', 'Match', 'Breath Of Heaven')
('Amy Grant', 'Match', 'Big Yellow Taxi')
('Amy Grant', 'Match', 'Lead Me On')
('Amy Grant', 'Match', 'Simple Things')
('Amy Grant', 'Match', 'Tennessee Christmas')
('Amy Rigby', 'Match', 'The Sugar Tree')
('Anacrusis', 'Match', 'Manic Impressions')
('Anal Cunt', 'Match', 'Everyone Should Be Killed')
('Anal Cunt', 'Match', 'I Like It When You Die')
('Anal Cunt', 'Match', 'It Just Gets Worse')
('Anal Cunt', 'Match', 'Top 40 Hits')
('Anasarca', 'Match', 'Moribund')
('Ancient', 'Match', 'Mad Grandiose Bloodfiends')
('Ancient', 'Match', 'Proxima Centauri')
('Ancient', 'Match', 'The Cainian Chronicle')
('Ancient', 'Match', 'The Halls Of Eternity')
('And Oceans', 'Match', 'The Symmetry Of I The Circle Of O')
('Andre Williams', 'Match', 'Silky')
('Andrea Marcovicci', 'Match', 'Marcovicci Sings Movies')
('Andy M. Stewart', 'Match', 'Fire In The Glen')
('Angelina Réaux', 'Match', 'Stranger Here Myself: Songs of Kurt Weill')
('Angizia', 'Match', 'Das Schachbrett Des Trommelbuben Zacharias')
('Angizia', 'Match', 'Die Kemenaten Scharlachroter Lichter')
('Ani DiFranco', 'Match', 'Little Plastic Castle')
('Anita Baker', 'Match', 'The Songstress')
('Anita Baker', 'Match', 'The Songstress')
('Anonymous 4', 'Match', "Love's Illusion")
('Anorexia Nervosa', 'Match', 'Exile')
('Anthony Moore', 'Match', "Flying Doesn't Help")
('Antichrisis', 'Match', 'Perfume')
('Anvil', 'Match', 'Absolutely No Alternative')
('Anvil', 'Match', 'Still Going Strong')
('Anvil', 'Match', 'This Is Thirteen')
('Anvil', 'Match', 'Backwaxed')
('Apoptygma Berzerk', 'Match', '7')
('Aqua', 'Match', 'Aquarius')
('Aqua', 'Match', 'Bubble Mix')
('Aqua', 'Match', 'Cartoon Heroes')
('Aqua', 'Match', 'Aquarium')
('Aqua', 'Match', 'Megalomania')
('Aqua', 'Match', 'Bumble Bees')
('Aqua', 'Match', 'Barbie Girl')
('Aqua', 'Match', 'My Oh My')
('Aqua', 'Match', 'Roses Are Red')
('Arctic Monkeys', 'Match', 'AM')
('Arctic Monkeys', 'Match', 'Favourite Worst Nightmare')
('Arctic Monkeys', 'Match', 'Humbug')
('Arctic Monkeys', 'Match', "Whatever People Say I Am, That's What I'm Not")
('Area 54', 'Match', 'No Visible Scars')
('Aretha Franklin', 'Match', 'Amazing Grace')
('Aretha Franklin', 'Match', 'I Knew You Were Waiting (For Me)')
('Aretha Franklin', 'Match', '30 Greatest Hits')
('Aretha Franklin', 'Match', "Aretha's Gold")
('Ariana Grande', 'Match', 'Yours Truly')
('Ariana Grande', 'Match', 'Santa Tell Me')
('Arkham 13', 'Match', 'The Wrath')
('Arma Angelus', 'Match', 'Where Sleeplessness Is Rest From Nightmares')
('Armored Saint', 'Match', 'Revelation')
('Armored Saint', 'Match', 'Nod To The Old School')
('Armored Saint', 'Match', 'Saints Will Conquer')
('Arthur Lyman', 'Match', 'Hawaiian Sunset')
('As All Die', 'Match', 'Time Of War And Conflict')
('As Friends Rust', 'Match', 'Won')
('As Friends Rust', 'Match', 'The Fists Of Time An Anthology Of Short Fiction And Non-Fiction')
('As Friends Rust', 'Match', 'As Friends Rust')
('Ashes', 'Match', 'And The Angels Wept')
('Ashes', 'Match', 'Death Has Made Its Call')
('Asia', 'Match', 'Aria')
('Asia', 'Match', 'Aura')
('Asia', 'Match', 'Aqua')
('Aspera Ad Astra', 'Match', 'Peace')
('Assemblage 23', 'Match', 'Contempt')
('Assemblage 23', 'Match', 'Failure')
('Assert', 'Match', 'Insurrection Rocks')
('Assück', 'Match', 'Misery Index')
('Astroqueen', 'Match', 'Into Submission')
('At The Drive-In', 'Match', 'Vaya')
('Athenaeum', 'Match', 'Radiance')
('Ativin', 'Match', 'Pills Versus Planes')
('Atlantic Starr', 'Match', 'Atlantic Starr')
('Atlantic Starr', 'Match', 'Brilliance')
('Atlantic Starr', 'Match', 'Legacy')
('Atlantic Starr', 'Match', 'Radiant')
('Atlantic Starr', 'Match', 'Time')
('Atlantic Starr', 'Match', "We're Movin' Up")
('Atlantic Starr', 'Match', 'Love Crazy')
('Atlantic Starr', 'Match', 'One Lover At A Time')
('Atlantic Starr', 'Match', 'Yours Forever')
('Atlantic Starr', 'Match', 'Ultimate Collection')
('Atlantis Ita', 'Match', 'The Good Land')
('Atomic Kitten', 'Match', 'Right Now')
('Atomic Opera', 'Match', 'Gospel Cola')
('Atrax Morgue', 'Match', 'Sickness Report')
('Atrox', 'Match', 'Contact')
('Atrox', 'Match', 'Terrestrials')
('Aurora', 'Match', 'Eos - Sadiam')
('Autograph', 'Match', 'Missing Pieces')
('Autonomy', 'Match', 'Autonomy')
('Avant', 'Match', 'My Thoughts')
('Avicii', 'Match', 'True')
('Avskum', 'Match', 'In The Spirit Of Mass Destruction')
('Ayahuasca', 'Match', 'Digital Alchemy')
('Annie Lennox', 'Match', 'A Christmas Cornucopia')
('Annie Lennox', 'Match', 'Medusa + Live In Central Park')
('Annie Lennox', 'Match', 'A Whiter Shade Of Pale')
('Annie Lennox', 'Match', 'Cold')
('Annie Lennox', 'Match', 'Little Bird')
('Annie Lennox', 'Match', 'The Annie Lennox Collection')
('Ace of Base', 'Match', 'Greatest Hits')
('Ace of Base', 'Match', 'Singles Of The 90s')
('Ace of Base', 'Match', 'The Collection')
('Ace of Base', 'Match', 'Happy Nation')
('Ace of Base', 'Match', 'The Golden Ratio')
('Ace of Base', 'Match', 'The Sign')
('Ace of Base', 'Match', 'Happy Nation')
('Armin Van Buuren', 'Match', '10 Years')
('Meat Loaf', 'Match', 'Bad Attitude')
('Meat Loaf', 'Match', 'Blind Before I Stop')
('Meat Loaf', 'Match', 'Dead Ringer')
('Meat Loaf', 'Match', 'Hang Cool Teddy Bear')
('Meat Loaf', 'Match', 'Hell in a Handbasket')
('Meat Loaf', 'Match', 'Midnight at the Lost and Found')
('B.J. Thomas', 'Match', 'Hooked On A Feeling: Greatest & Latest')
('Babylonian Tiles', 'Match', 'Teknicolour Aftermath')
('Backfire!', 'Match', 'Still Dedicated')
('Backstreet Boys', 'Match', "Backstreet's Back")
('Backstreet Boys', 'Match', 'Never Gone')
('Backstreet Boys', 'Match', 'Unbreakable')
('Backstreet Boys', 'Match', 'Selections From A Night Out With The Backstreet Boys')
('Bad Manners', 'Match', 'Fat Sound')
('Bad Manners', 'Match', 'Heavy Petting')
('Bad Times', 'Match', 'Bad Times')
('Bad Trip', 'Match', 'Fear And Loathing')
('Badfinger', 'Match', 'Head First')
('Bagman', 'Match', 'Wrap')
('Balaam And The Angel', 'Match', 'Prime Time')
('Banana Erectors', 'Match', 'Banana Erectors')
('Bane', 'Match', 'Give Blood')
('Bane', 'Match', 'Holding This Moment')
('Bang Tango', 'Match', 'Live')
('Barenaked Ladies', 'Match', 'Gordon')
('Barenaked Ladies', 'Match', 'Rock Spectacle')
('Barenaked Ladies', 'Match', 'Pinch Me')
('Barenaked Ladies', 'Match', 'Disc One: All Their Greatest Hits (1991-2001)')
('Barracudas', 'Match', 'Drop Out With The Barracudas')
('Barracudas', 'Match', 'Through The Mysts Of Time')
('Barry White', 'Match', 'Barry & Glodean')
('Barry White', 'Match', 'Barry White The Man')
('Barry White', 'Match', 'Beware!')
('Barry White', 'Match', 'Dedicated')
('Barry White', 'Match', 'Is This Whatcha Wont?')
('Barry White', 'Match', "Stone Gon'")
('Barry White', 'Match', 'The Icon Is Love')
('Barry White', 'Match', 'The Message Is Love')
('Barry White', 'Match', 'The Right Night & Barry White')
('Barry White', 'Match', 'Gold - The Very Best Of')
('Barry White', 'Match', 'Just For You')
('Barry White', 'Match', 'Love Songs')
('Barry White', 'Match', 'Change')
('Barry White', 'Match', 'Sheet Music')
('Barry White', 'Match', 'Staying Power')
('Barry White', 'Match', 'Under The Influence Of Love')
('Bassholes', 'Match', 'The Secret Strength Of Depression (Live At KSPC, Claremont)')
('Beastie Boys', 'Match', 'Hot Sauce Committee Part Two')
('Beastie Boys', 'Match', 'Ill Communication')
('Beastie Boys', 'Match', "Paul's Boutique")
('Beastie Boys', 'Match', 'The Mix-Up')
('Beaver Nelson', 'Match', 'Little Brother')
('Beck', 'Match', 'Golden Feelings')
('Beck', 'Match', 'Guerolito')
('Beck', 'Match', 'Modern Guilt')
('Beck', 'Match', 'Guero')
('Beck', 'Match', 'Morning Phase')
('Beenie Man', 'Match', 'Beenie Man Meets Mad Cobra')
('Beenie Man', 'Match', 'Cool Cool Rider')
('Beenie Man', 'Match', 'Dis Unu Fi Hear')
('Beenie Man', 'Match', 'Gold')
('Beenie Man', 'Match', 'Many Moods Of Moses')
('Beenie Man', 'Match', 'Tropical Storm')
('Beenie Man', 'Match', 'Undisputed')
('Beenie Man', 'Match', 'Unstoppable')
('Beenie Man', 'Match', "Ruff 'N' Tuff")
('Beenie Man', 'Match', 'The Magnificent')
('Beenie Man', 'Match', 'Y2K')
('Beenie Man', 'Match', 'Youth Quake')
('Beenie Man', 'Match', 'Blessed')
('Beenie Man', 'Match', 'Defend It')
('Beenie Man', 'Match', 'Maestro')
('Beenie Man', 'Match', 'The Doctor')
('Beezus', 'Match', 'Breakfast Was Weird')
('Beherit', 'Match', 'Messe Des Morts')
('Bei Maejor', 'Match', 'Upside Down Vol 2')
('Belphegor', 'Match', 'Necrodaemon Terrorsathan')
('Ben Folds', 'Match', 'Ben Folds Live')
('Ben Folds', 'Match', 'Supersunnyspeedgraphic, The LP')
('Ben Folds', 'Match', 'Speed Graphic')
('Ben Folds', 'Match', 'Sunny 16')
('Ben Folds', 'Match', 'Super D')
('Ben Harper', 'Match', 'Get Up!')
('Ben Harper', 'Match', "Give Till It's Gone")
('Ben Watt', 'Match', 'North Marine Drive + Summer Into Winter')
('Benjamin B.', 'Match', 'Forever Careless')
('Benny Joy', 'Match', '"Rockin\' And Rollin\' With Benny Joy')
('Bent Leg Fatima', 'Match', 'Bent Leg Fatima')
('Berlin', 'Match', 'Best Of Berlin 1979 - 1988')
('Bert Jansch', 'Match', 'Toy Balloon')
('Beseech', 'Match', 'Black Emotions')
('Beth Capper', 'Match', 'Complimentary Mood Enhancer (Presents The Original Soundtrack For The Motion Picture "Wig-Lickers" Based On The True Story Of Trigger)')
('Beth Hart', 'Match', 'Live At The Royal Albert Hall')
('Beth Hart', 'Match', 'Bang Bang Boom Boom')
('Beth Hart', 'Match', '37 Days')
('Bethzaida', 'Match', 'LXXVIII')
('Bethzaida', 'Match', 'Nine Worlds')
('Bette Midler', 'Match', 'Beaches (Original Soundtrack Recording)')
('Better Than Ezra', 'Match', 'All Together Now')
('Better Than Ezra', 'Match', 'Closer')
('Better Than Ezra', 'Match', 'Paper Empire')
('Better Than Ezra', 'Match', 'Artifakt')
('Better Than Ezra', 'Match', 'Greatest Hits')
('Beulah', 'Match', 'When Your Heartstrings Break')
('Beyond Dawn', 'Match', 'Pity Love')
('Beyond', 'Match', 'Reassemble')
('Big Ass Truck', 'Match', 'The Rug')
('Big Brother & The Holding Company', 'Match', "Can't Go Home Again")
('Big Brother & The Holding Company', 'Match', 'Do What You Love')
('Big Electric Cat', 'Match', 'Eyelash')
('Big K.R.I.T.', 'Match', "It's Better This Way")
('Big L', 'Match', 'Deadly Combination')
('Blackstreet', 'Match', 'Another Level')
('Boyz II Men', 'Match', 'Evolution')
('Big Head Todd', 'Match', 'Sister Sweetly')
('Carpenters', 'Match', 'Now & Then')
('Count Basie', 'Match', 'Basie Jam #3')
('Dave Brubeck', 'Match', 'Time In')
('DJ Shadow', 'Match', "This Time (I'm Gonna Try It My Way)")
('DJ Shadow', 'Match', "Mashin' On The Motorway")
('Dr.Alban', 'Match', "It's My Life")
('David Cassidy', 'Match', 'Then and Now')
('Edguy', 'Match', 'Theater of Salvation')
('Eisbrecher', 'Match', 'Die Hölle Muss Warten')
('Fatboy Slim', 'Match', 'The Rockafeller Skank')
('Galactic Cowboys', 'Match', 'Let It Go')
('Galactic Cowboys', 'Match', 'Machine Fish')
('Galactic Cowboys', 'Match', 'The Horse That Bud Bought')
('Glen Hansard', 'Match', 'Rhythm and Repose')
('Ginuwine', 'Match', 'Greatest Hits')
('Herbie Hancock', 'Match', 'Maiden Voyage')
('INXS', 'Match', 'Never Tear Us Apart')
('Jaci Velasquez', 'Match', 'Jaci Velasquez')
('Jimmy Eat World', 'Match', 'Jimmy Eat World')
('Kanye West', 'Match', 'Stronger')
('Kim Wilde', 'Match', 'The Singles Collection 1981-1993')
('Kim Wilde', 'Match', 'The Very Best of Kim Wilde')
('Lenny Kravitz', 'Match', 'Greatest Hits')
('Luke Combs', 'Match', 'What You See Is What You Get')
('Lil Uzi Vert', 'Match', 'Luv Is Rage 2')
('Maxx', 'Match', 'Get-A-Way')
('Michael W. Smith', 'Match', 'Christmas')
('Michael W. Smith', 'Match', 'Worship')
('Muse', 'Match', 'HAARP')
('Muse', 'Match', 'Butterflies & Hurricanes')
('Muse', 'Match', 'Sing For Absolution')
('Muse', 'Match', 'Hullabaloo Soundtrack')
('Nazareth', 'Match', 'Nazareth')
('Nazareth', 'Match', 'Greatest Hits')
('Nickelback', 'Match', "Savin' Me")
('Notorious B.I.G.', 'Match', 'Born Again')
('Notorious B.I.G.', 'Match', 'Duets: The Final Chapter')
('New Found Glory', 'Match', 'Makes Me Sick')
('Pantera', 'Match', 'Planet Caravan')
('Paul McCartney', 'Match', 'Run Devil Run')
('Pink Floyd', 'Match', 'Relics')
('Reba McEntire', 'Match', 'If You See Him')
('Reba McEntire', 'Match', 'Reba McEntire')
('Reba McEntire', 'Match', 'So Good Together')
('Reba McEntire', 'Match', 'The Last One to Know')
('Reba McEntire', 'Match', 'Unlimited')
('Red Hot Chili Peppers', 'Match', 'Californication')
('Rick Astley', 'Match', 'Whenever You Need Somebody')
('Robbie Williams', 'Match', 'Rudebox')
('Ryan Adams', 'Match', '29')
('Ryan Adams', 'Match', 'Demolition')
('Ryan Adams', 'Match', 'Easy Tiger')
('Ryan Adams', 'Match', 'Gold')
('Ryan Adams', 'Match', 'Heartbreaker')
('Seal', 'Match', 'Seal')
('Slayer', 'Match', 'Show No Mercy')
('Sophie B. Hawkins', 'Match', 'Whaler')
('Staind', 'Match', '14 Shades of Grey')
('Staind', 'Match', 'Break the Cycle')
('Sum 41', 'Match', 'Underclass Hero')
('Tracy Chapman', 'Match', 'New Beginning')
('Tim McGraw', 'Match', 'Number One Hits')
('Trace Adkins', 'Match', "Something's Going On")
('The Band Perry', 'Match', 'Pioneer')
('The Manhattan Transfer', 'Match', 'Bop Doo-Wopp')
('Vince Guaraldi Trio', 'Match', 'A Charlie Brown Christmas')
('Vampire Weekend', 'Match', 'Holiday')
('Vengaboys', 'Match', 'Boom, Boom, Boom, Boom!!')
('W.A.S.P.', 'Match', 'Helldorado')
('W.A.S.P.', 'Match', 'Kill Fuck Die')
('Wayne Shorter', 'Match', 'Atlantis')
('Wayne Shorter', 'Match', 'Native Dancer')
('Wayne Shorter', 'Match', 'Phantom Navigator')
"""